# EXEMPLO 3: Conversational RAG Pipeline — Groq + LangChain + BGE-M3 (Ollama)
## OpenSearch Hybrid Search com memória de contexto + auto-ingestão do corpus TCU 2026

**Stack alinhada ao curso (Aula 3+):**
- **LLM**: Groq Cloud (`llama-3.1-8b-instant`) via `langchain_groq.ChatGroq` — fallback automático `langchain_ollama.ChatOllama` (`llama3.2:3b`)
- **Embeddings**: BGE-M3 via Ollama (`bge-m3`, dim=1024) usando `langchain_ollama.OllamaEmbeddings`
- **Retrieval**: Hybrid Search (BM25 + kNN) no OpenSearch 3.x
- **Dataset**: `aula4/datasets/corpus_juridico_aula4_v2.json` (1.100 acórdãos TCU 2026) — fallback para o corpus legado pequeno

## Objetivo

Construir um pipeline conversacional multi-turno auto-contido que:
1. **Cria (ou recria) o índice** no OpenSearch e **ingere o corpus jurídico**
2. Executa **retrieval híbrido** (BM25 + Vector) sobre o corpus indexado
3. **Gera respostas** via Groq (alta velocidade) com fallback Ollama local
4. Mantém **memória de conversa** com histórico estruturado

## Pré-requisitos

1. **Variáveis em `.env`** (ver Aula 1 / Aula 3):
   ```
   GROQ_API_KEY=gsk_...
   GROQ_LLM_MODEL=llama-3.1-8b-instant
   OLLAMA_BASE_URL=http://localhost:11434
   OLLAMA_LLM_MODEL=llama3.2:3b
   OLLAMA_EMBED_MODEL=bge-m3
   OPENSEARCH_HOST=localhost
   OPENSEARCH_PORT=9200
   OPENSEARCH_USER=admin
   OPENSEARCH_PASSWORD=admin
   ```
2. **Ollama** rodando com `bge-m3` carregado (`ollama pull bge-m3`)
3. **OpenSearch 3.x** acessível

## 1. Instalação de Dependências

Pacotes LangChain explícitos para Groq e Ollama. Sem estes pacotes os imports `langchain_groq` e `langchain_ollama` falham.

In [1]:
import subprocess, sys

packages = [
    'opensearch-py>=2.7',
    'requests>=2.31.0',
    'python-dotenv>=1.0.0',
    'pandas>=2.0.0',
    'tqdm>=4.66',
    # LangChain core + integrações específicas
    'langchain>=0.2.0',
    'langchain-core>=0.2.0',
    'langchain-community>=0.2.0',
    'langchain-groq>=0.1.0',     # ChatGroq (LLM Groq)
    'langchain-ollama>=0.1.0',   # ChatOllama + OllamaEmbeddings
    'langchain-openai>=0.1.0',   # opcional — endpoint OpenAI-compatible
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✓ Dependências instaladas:')
for p in packages:
    print(f'   - {p}')

✓ Dependências instaladas:
   - opensearch-py>=2.7
   - requests>=2.31.0
   - python-dotenv>=1.0.0
   - pandas>=2.0.0
   - tqdm>=4.66
   - langchain>=0.2.0
   - langchain-core>=0.2.0
   - langchain-community>=0.2.0
   - langchain-groq>=0.1.0
   - langchain-ollama>=0.1.0
   - langchain-openai>=0.1.0


## 2. Carregar `.env` (Groq API key, Ollama URL, OpenSearch creds)

`load_dotenv()` lê o arquivo `.env` da pasta atual (ou de `~/mba-rag/.env`) e injeta no `os.environ`. Sem isso, `GROQ_API_KEY` não fica disponível e o fallback Ollama é acionado mesmo com Groq configurado.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_CANDIDATES = [
    Path.cwd() / '.env',                       # ./.env (pasta do notebook)
    Path.home() / 'mba-rag' / '.env',          # ~/mba-rag/.env (padrão Aula 1)
    Path.cwd().parent.parent / '.env',         # raiz do projeto MBA_RAG_CAG
]

loaded_from = None
for env_path in ENV_CANDIDATES:
    if env_path.exists():
        load_dotenv(dotenv_path=env_path, override=True)
        loaded_from = env_path
        break
else:
    load_dotenv(override=False)

if loaded_from:
    print(f'✓ .env carregado de: {loaded_from}')
else:
    print('⚠️  Nenhum arquivo .env encontrado. Defina manualmente as variáveis abaixo.')

for k in ['GROQ_API_KEY', 'GROQ_LLM_MODEL',
          'OLLAMA_BASE_URL', 'OLLAMA_LLM_MODEL', 'OLLAMA_EMBED_MODEL',
          'OPENSEARCH_HOST', 'OPENSEARCH_PORT']:
    v = os.getenv(k, '')
    show = ('***' + v[-4:]) if (k == 'GROQ_API_KEY' and v) else v
    print(f'  {k:22s} = {show or "(não definida)"}')

✓ .env carregado de: d:\IBMEC\MBA_RAG_CAG\.env
  GROQ_API_KEY           = ***ZnFy
  GROQ_LLM_MODEL         = llama-3.1-8b-instant
  OLLAMA_BASE_URL        = http://localhost:11434
  OLLAMA_LLM_MODEL       = (não definida)
  OLLAMA_EMBED_MODEL     = bge-m3:latest
  OPENSEARCH_HOST        = localhost
  OPENSEARCH_PORT        = 9200


## 3. Imports — LangChain + OpenSearch

In [4]:
import json, time, warnings
from datetime import datetime
from typing import Dict, List, Any, Optional

import pandas as pd
from tqdm.auto import tqdm
from opensearchpy import OpenSearch

# ── LangChain — Groq (LLM primário) ───────────────────────────────────────
from langchain_groq import ChatGroq
# ── LangChain — Ollama (LLM fallback + embeddings BGE-M3) ─────────────────
from langchain_ollama import ChatOllama, OllamaEmbeddings
# ── LangChain — Core (prompts, mensagens) ─────────────────────────────────
from langchain_core.messages       import HumanMessage, SystemMessage
from langchain_core.prompts        import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

warnings.filterwarnings('ignore')

def log(msg: str, level: str = 'INFO'):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {level}: {msg}")

log('LangChain + integrações Groq/Ollama importadas com sucesso')

d:\IBMEC\MBA_RAG_CAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[10:50:05] INFO: LangChain + integrações Groq/Ollama importadas com sucesso


## 4. Conexão, Embeddings e Auto-ingestão do Corpus TCU 2026

**Comportamento da célula** (idempotente — pode ser re-executada):
1. Conecta no OpenSearch e instancia `OllamaEmbeddings` (BGE-M3, 1024d)
2. **Se o índice já existir → APAGA e RECRIA** (garante mapping correto e dados consistentes)
3. **Se não existir → CRIA** com mapping híbrido (knn_vector + text português)
4. Carrega `aula4/datasets/corpus_juridico_aula4_v2.json` (1.100 acórdãos TCU 2026); fallback para o corpus legado pequeno
5. Gera embeddings em **batch via `OllamaEmbeddings.embed_documents`** e indexa em **bulk** no OpenSearch
6. Atualiza o índice (`refresh`) e confirma a contagem

In [5]:
# ── 4.1 Conexão OpenSearch ────────────────────────────────────────────────
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST',     'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER',     'admin')
OPENSEARCH_PASS = os.getenv('OPENSEARCH_PASSWORD', 'admin')
INDEX_NAME      = os.getenv('INDEX_NAME', 'exemplo3_corpus_juridico_tcu')

client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, timeout=60,
)
info = client.info()
log(f"OpenSearch {info['version']['number']} OK → índice alvo: {INDEX_NAME}")

# ── 4.2 Embeddings via langchain_ollama.OllamaEmbeddings ──────────────────
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',    'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
EMBED_DIM = 1024

embeddings = OllamaEmbeddings(
    model    = OLLAMA_EMBED_MODEL,
    base_url = OLLAMA_BASE_URL,
)
_v = embeddings.embed_query('teste de embedding')
assert len(_v) == EMBED_DIM, f'Dim inesperada: {len(_v)}'
log(f'OllamaEmbeddings OK → modelo={OLLAMA_EMBED_MODEL}, dim={len(_v)}')

# ── 4.3 Mapping do índice híbrido (kNN + BM25 em português) ───────────────
INDEX_MAPPING = {
    'settings': {
        'index': {
            'knn': True,
            'knn.algo_param.ef_search': 100,
            'number_of_shards':   1,
            'number_of_replicas': 0,
        },
        'analysis': {
            'analyzer': {
                'portuguese_analyzer': {'type': 'standard', 'stopwords': '_portuguese_'}
            }
        }
    },
    'mappings': {
        'properties': {
            'id':        {'type': 'keyword'},
            'tipo':      {'type': 'keyword'},
            'titulo':    {'type': 'text', 'analyzer': 'portuguese_analyzer'},
            'conteudo':  {'type': 'text', 'analyzer': 'portuguese_analyzer'},
            'metadata':  {'type': 'object', 'enabled': True},
            'embedding': {
                'type':      'knn_vector',
                'dimension': EMBED_DIM,
                'method': {
                    'name':       'hnsw',
                    'space_type': 'cosinesimil',
                    'engine':     'faiss',
                    'parameters': {'ef_construction': 128, 'm': 16}
                }
            }
        }
    }
}

# ── 4.4 Recriação idempotente do índice ───────────────────────────────────
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    log(f'Índice anterior "{INDEX_NAME}" APAGADO (já existia)')
else:
    log(f'Índice "{INDEX_NAME}" não existia')

client.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
log(f'Índice "{INDEX_NAME}" CRIADO (dim={EMBED_DIM}, hnsw/faiss/cosinesimil)')

[10:50:13] INFO: OpenSearch 3.0.0 OK → índice alvo: exemplo3_corpus_juridico_tcu
[10:50:19] INFO: OllamaEmbeddings OK → modelo=bge-m3:latest, dim=1024
[10:50:19] INFO: Índice "exemplo3_corpus_juridico_tcu" não existia
[10:50:19] INFO: Índice "exemplo3_corpus_juridico_tcu" CRIADO (dim=1024, hnsw/faiss/cosinesimil)


In [6]:
# ── 4.5 Carregar o dataset da Aula 4 ──────────────────────────────────────
CORPUS_PRIM = Path('..') / 'datasets' / 'corpus_juridico_aula4_v2.json'
CORPUS_LEG  = Path('..') / 'datasets' / 'corpus_juridico_aula4.json'

if CORPUS_PRIM.exists():
    corpus_path = CORPUS_PRIM
    LIMITE_DOCS = int(os.getenv('EXEMPLO3_MAX_DOCS', 200))  # subconjunto para o exemplo
elif CORPUS_LEG.exists():
    corpus_path = CORPUS_LEG
    LIMITE_DOCS = 9999
else:
    raise FileNotFoundError(
        f'Nenhum corpus encontrado em {CORPUS_PRIM} ou {CORPUS_LEG}. '
        f'Execute o pipeline da Aula 4 para gerar o corpus.')

with open(corpus_path, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

if len(corpus) > LIMITE_DOCS:
    log(f'Corpus tem {len(corpus)} docs — limitando a {LIMITE_DOCS} para este exemplo')
    corpus = corpus[:LIMITE_DOCS]

log(f'Corpus carregado: {corpus_path.name} → {len(corpus)} documentos')
log(f'Exemplo de doc:\n  id     : {corpus[0]["id"]}\n'
    f'  título : {corpus[0].get("titulo", "(s/título)")[:90]}\n'
    f'  conteúdo: {corpus[0].get("conteudo", "")[:120]}...')

[10:53:36] INFO: Corpus tem 1100 docs — limitando a 200 para este exemplo
[10:53:36] INFO: Corpus carregado: corpus_juridico_aula4_v2.json → 200 documentos
[10:53:36] INFO: Exemplo de doc:
  id     : acordao_2026_2344_2764954
  título : ACÓRDÃO DE RELAÇÃO 2344/2026 ATA 15/2026 - SEGUNDA CÂMARA
  conteúdo: ASSUNTO: REPRESENTAÇÃO. Representação referente operação de crédito de R$ 30.000.000,00 com garantia integral da União.
...


In [7]:
# ── 4.6 Geração de embeddings (batch) + Bulk indexing ─────────────────────
BATCH = 16   # tamanho do lote enviado ao Ollama por chamada

def chunked(it, n):
    for i in range(0, len(it), n):
        yield it[i:i + n]

log(f'Gerando embeddings BGE-M3 (Ollama) e indexando {len(corpus)} docs em lotes de {BATCH}...')
t0 = time.time()
erros = 0

for lote in tqdm(list(chunked(corpus, BATCH)), desc='Bulk ingest'):
    textos = [(d.get('titulo', '') + '\n\n' + d.get('conteudo', '')).strip() for d in lote]
    # OllamaEmbeddings.embed_documents → lista de vetores 1024d, na ordem
    vetores = embeddings.embed_documents(textos)

    bulk_body = []
    for doc, vec in zip(lote, vetores):
        bulk_body.append({'index': {'_index': INDEX_NAME, '_id': doc['id']}})
        bulk_body.append({
            'id':        doc['id'],
            'tipo':      doc.get('tipo', ''),
            'titulo':    doc.get('titulo', ''),
            'conteudo':  doc.get('conteudo', ''),
            'metadata':  doc.get('metadata', {}),
            'embedding': vec,
        })
    resp = client.bulk(body=bulk_body, request_timeout=180)
    if resp.get('errors'):
        erros += sum(1 for it in resp['items'] if it.get('index', {}).get('error'))

client.indices.refresh(index=INDEX_NAME)
elapsed = time.time() - t0
count = client.count(index=INDEX_NAME)['count']
log(f'Ingestão concluída em {elapsed:.1f}s | docs no índice: {count} | erros: {erros}')
assert count > 0, 'Nenhum documento indexado — verifique o Ollama e o OpenSearch.'

[10:53:48] INFO: Gerando embeddings BGE-M3 (Ollama) e indexando 200 docs em lotes de 16...


Bulk ingest: 100%|██████████| 13/13 [31:15<00:00, 144.27s/it]


ConnectionError: ConnectionError(HTTPConnection(host='localhost', port=9200): Failed to establish a new connection: [WinError 10061] Nenhuma conexão pôde ser feita porque a máquina de destino as recusou ativamente) caused by: NewConnectionError(HTTPConnection(host='localhost', port=9200): Failed to establish a new connection: [WinError 10061] Nenhuma conexão pôde ser feita porque a máquina de destino as recusou ativamente)

## 5. LLM — `ChatGroq` (primário) + `ChatOllama` (fallback)

Snippet padrão do curso (Aula 3+): tenta Groq via **`langchain_groq.ChatGroq`** com smoke test, faz fallback automático para **`langchain_ollama.ChatOllama`** se a Groq API falhar ou se `GROQ_API_KEY` estiver ausente.

In [8]:
GROQ_API_KEY     = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL   = os.getenv('GROQ_LLM_MODEL',   'llama-3.1-8b-instant')
OLLAMA_LLM_MODEL = os.getenv('OLLAMA_LLM_MODEL', 'llama3.2:3b')

llm = None
MODEL_NAME, LLM_PROVIDER = None, None

# 1) Tenta Groq via langchain_groq
if GROQ_API_KEY:
    try:
        candidate = ChatGroq(
            model       = GROQ_LLM_MODEL,
            api_key     = GROQ_API_KEY,
            temperature = 0.1,
            max_tokens  = 1024,
            timeout     = 30,
        )
        _ = candidate.invoke([HumanMessage(content='ok')])  # smoke test
        llm, MODEL_NAME, LLM_PROVIDER = candidate, GROQ_LLM_MODEL, 'groq'
        log(f'LLM primário: Groq via ChatGroq ({GROQ_LLM_MODEL})')
    except Exception as e:
        log(f'Groq indisponível ({e.__class__.__name__}: {e}). Fallback → Ollama.', 'WARN')

# 2) Fallback Ollama via langchain_ollama
if llm is None:
    llm = ChatOllama(
        model       = OLLAMA_LLM_MODEL,
        base_url    = OLLAMA_BASE_URL,
        temperature = 0.1,
        num_predict = 1024,
    )
    MODEL_NAME, LLM_PROVIDER = OLLAMA_LLM_MODEL, 'ollama'
    log(f'LLM fallback: Ollama via ChatOllama ({OLLAMA_LLM_MODEL})')

log(f'Provedor ativo: {LLM_PROVIDER} | modelo: {MODEL_NAME}')
_parser = StrOutputParser()

[11:31:06] INFO: LLM primário: Groq via ChatGroq (llama-3.1-8b-instant)
[11:31:06] INFO: Provedor ativo: groq | modelo: llama-3.1-8b-instant


## 6. Busca Híbrida (BM25 + BGE-M3 kNN)

In [9]:
def hybrid_search(query_text: str, index_name: str = INDEX_NAME, size: int = 5) -> List[Dict]:
    """Busca híbrida (BM25 + kNN) no OpenSearch usando embeddings BGE-M3 via Ollama."""
    q_emb = embeddings.embed_query(query_text)
    body = {
        'size': size,
        'query': {
            'bool': {
                'should': [
                    {'multi_match': {'query': query_text,
                                       'fields': ['titulo^2', 'conteudo']}},
                    {'knn': {'embedding': {'vector': q_emb, 'k': size}}},
                ]
            }
        }
    }
    try:
        r = client.search(index=index_name, body=body)
        return [{'id': h['_id'], 'score': h['_score'], 'source': h['_source']}
                for h in r['hits']['hits']]
    except Exception as e:
        log(f'Erro busca: {e}', 'ERROR')
        return []

# Smoke test da busca
_hits = hybrid_search('operação de crédito com garantia da União', size=3)
print(f'\nSmoke test — top {len(_hits)} resultados:')
for h in _hits:
    print(f"  [{h['score']:.3f}] {h['source'].get('titulo', h['id'])[:90]}")


Smoke test — top 3 resultados:
  [7.373] ACÓRDÃO DE RELAÇÃO 2344/2026 ATA 15/2026 - SEGUNDA CÂMARA
  [5.304] ACÓRDÃO DE RELAÇÃO 2343/2026 ATA 15/2026 - SEGUNDA CÂMARA
  [3.894] ACÓRDÃO 1285/2026 ATA 17/2026 - PLENÁRIO


## 7. Pipeline Conversacional (LCEL)

Combina **histórico de turnos** + **busca híbrida** + **`ChatPromptTemplate`** (LCEL) para gerar respostas com contexto.

In [10]:
PROMPT_TEMPLATE = ChatPromptTemplate.from_messages([
    ('system',
     'Você é um assistente jurídico especializado em acórdãos do TCU e direito brasileiro. '
     'Responda baseado SOMENTE nos documentos fornecidos e cite-os (Doc 1, Doc 2, ...). '
     'Se a informação não estiver nos documentos, diga claramente.'),
    ('human',
     'Histórico recente da conversa:\n{historico}\n\n'
     'Documentos recuperados:\n{contexto}\n\n'
     'Pergunta atual: {pergunta}\n\n'
     'Responda de forma objetiva.')
])

CHAIN = PROMPT_TEMPLATE | llm | _parser   # cadeia LangChain LCEL

class ConversationalRAG:
    def __init__(self, index_name: str = INDEX_NAME, k_retrieval: int = 5):
        self.index_name = index_name
        self.k          = k_retrieval
        self.history: List[Dict[str, Any]] = []
        self.turn       = 0

    def _format_history(self, last_n: int = 3) -> str:
        if not self.history:
            return '(sem histórico)'
        recent = self.history[-last_n:]
        return '\n'.join(f"Pergunta: {t['user']}\nResposta: {t['assistant']}" for t in recent)

    def _format_context(self, results: List[Dict]) -> str:
        if not results:
            return '(nenhum documento recuperado)'
        chunks = []
        for i, r in enumerate(results[: self.k], 1):
            src    = r['source']
            titulo = src.get('titulo') or src.get('id', '')
            texto  = src.get('conteudo', '')
            chunks.append(f"[Doc {i}] {titulo}\n{texto[:600]}")
        return '\n\n'.join(chunks)

    def ask(self, user_query: str) -> Dict[str, Any]:
        self.turn += 1
        t0 = time.perf_counter()

        results   = hybrid_search(user_query, self.index_name, size=self.k)
        contexto  = self._format_context(results)
        historico = self._format_history(last_n=3)

        resposta = CHAIN.invoke({
            'historico': historico,
            'contexto':  contexto,
            'pergunta':  user_query,
        }).strip()

        elapsed_ms = (time.perf_counter() - t0) * 1000
        turn_obj = {
            'turn':         self.turn,
            'timestamp':    datetime.now().isoformat(),
            'user':         user_query,
            'assistant':    resposta,
            'retrieved':    [r['id'] for r in results],
            'latency_ms':   round(elapsed_ms, 1),
            'llm_provider': LLM_PROVIDER,
            'llm_model':    MODEL_NAME,
        }
        self.history.append(turn_obj)
        return turn_obj

    def history_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.history) if self.history else pd.DataFrame()

rag = ConversationalRAG(index_name=INDEX_NAME, k_retrieval=5)
log(f'Pipeline conversacional inicializado em {INDEX_NAME} | LLM={LLM_PROVIDER}/{MODEL_NAME}')

[11:32:32] INFO: Pipeline conversacional inicializado em exemplo3_corpus_juridico_tcu | LLM=groq/llama-3.1-8b-instant


## 8. Turno 1 — Consulta Inicial (domínio TCU)

In [11]:
t1 = rag.ask('O que caracteriza uma operação de crédito com garantia da União e quais irregularidades costumam ser apuradas pelo TCU?')
print(f"Turno {t1['turn']} | Provedor: {t1['llm_provider']} ({t1['llm_model']}) | {t1['latency_ms']} ms")
print(f"Docs recuperados: {t1['retrieved']}\n")
print(t1['assistant'])

Turno 1 | Provedor: groq (llama-3.1-8b-instant) | 3789.4 ms
Docs recuperados: ['acordao_2026_2344_2764954', 'acordao_2026_2343_2765696', 'acordao_2026_1272_2766269', 'acordao_2026_1285_2765506', 'acordao_2026_1192_2701361']

Com base nos documentos fornecidos, posso responder à sua pergunta.

Uma operação de crédito com garantia da União é caracterizada por um contrato em que o governo federal assume a responsabilidade de garantir o pagamento de uma dívida, geralmente em caso de inadimplência do tomador do crédito. Isso é comum em operações de crédito realizadas por instituições financeiras, como bancos, para financiar projetos ou atividades de empresas ou entidades públicas.

Quanto às irregularidades que costumam ser apuradas pelo TCU em operações de crédito com garantia da União, os documentos fornecidos não mencionam especificamente quais são as irregularidades mais comuns. No entanto, podemos inferir que o TCU pode apurar irregularidades relacionadas à:

* Desvio de finalidade: qu

## 9. Turno 2 — Follow-up com Memória

In [12]:
t2 = rag.ask('Quando o TCU pode determinar medida cautelar em representações sobre essas operações?')
print(f"Turno {t2['turn']} | Provedor: {t2['llm_provider']} | {t2['latency_ms']} ms\n")
print(t2['assistant'])

Turno 2 | Provedor: groq | 3415.0 ms

Com base nos documentos fornecidos, posso responder à sua pergunta.

O TCU pode determinar medida cautelar em representações sobre operações de crédito com garantia da União em casos em que há indícios de irregularidades graves e iminentes, que possam causar danos irreparáveis à administração pública ou ao erário.

Isso é mencionado no Acórdão 2344/2026 (Doc 2), que trata de uma representação formulada em face de possíveis irregularidades na execução do Contrato BB 40/00028-1, no valor de R$ 30.000.000,00. Nesse caso, o TCU determinou medida cautelar para evitar a continuidade de possíveis irregularidades e garantir a regularidade da gestão dos recursos.

Além disso, o Acórdão 1286/2026 (Doc 3) também menciona a possibilidade de determinação de medida cautelar em casos de irregularidades graves e iminentes, como a ausência de previsão editálica de sistema de registro de preços e a desclassificação de empresas por ofertar taxa negativa ou desconto.


## 10. Turno 3 — Pergunta cruzada

In [13]:
t3 = rag.ask('Resuma em três pontos os principais aspectos das duas perguntas anteriores.')
print(f"Turno {t3['turn']} | {t3['latency_ms']} ms\n")
print(t3['assistant'])

Turno 3 | 3750.5 ms

Com base nos documentos fornecidos, posso resumir os principais aspectos das duas perguntas anteriores em três pontos:

1. **Operações de crédito com garantia da União**: Uma operação de crédito com garantia da União é caracterizada por um contrato em que o governo federal assume a responsabilidade de garantir o pagamento de uma dívida, geralmente em caso de inadimplência do tomador do crédito. O TCU pode apurar irregularidades relacionadas à desvio de finalidade, triangulação bancária, fraude em medições e outras irregularidades relacionadas à legalidade e regularidade da operação.
2. **Medida cautelar do TCU**: O TCU pode determinar medida cautelar em representações sobre operações de crédito com garantia da União em casos em que há indícios de irregularidades graves e iminentes que possam causar danos irreparáveis à administração pública ou ao erário. Isso é mencionado nos Acórdãos 2344/2026 (Doc 1) e 1286/2026 (Doc 3).
3. **Documentos fornecidos**: Os documento

## 11. Histórico da Conversa

In [14]:
if rag.history:
    df = rag.history_df()
    print(f"Total de turnos: {len(df)}")
    cols = ['turn', 'user', 'latency_ms', 'llm_provider', 'llm_model']
    print('\nResumo:')
    print(df[cols].to_string(index=False))
else:
    print('Sem histórico ainda.')

Total de turnos: 3

Resumo:
 turn                                                                                                                    user  latency_ms llm_provider            llm_model
    1 O que caracteriza uma operação de crédito com garantia da União e quais irregularidades costumam ser apuradas pelo TCU?      3789.4         groq llama-3.1-8b-instant
    2                                   Quando o TCU pode determinar medida cautelar em representações sobre essas operações?      3415.0         groq llama-3.1-8b-instant
    3                                             Resuma em três pontos os principais aspectos das duas perguntas anteriores.      3750.5         groq llama-3.1-8b-instant


## Referências (ABNT)

GROQ INC. **Groq API Documentation**. Disponível em: <https://console.groq.com/docs>.

LANGCHAIN. **`langchain-groq` integration**. Disponível em: <https://python.langchain.com/docs/integrations/chat/groq/>.

LANGCHAIN. **`langchain-ollama` integration**. Disponível em: <https://python.langchain.com/docs/integrations/chat/ollama/>.

OLLAMA. **Ollama BGE-M3 model**. Disponível em: <https://ollama.com/library/bge-m3>.

OPENSEARCH PROJECT. **Hybrid Search**. 3.0 Docs. Disponível em: <https://docs.opensearch.org/3.0/vector-search/ai-search/hybrid-search/>.

TRIBUNAL DE CONTAS DA UNIÃO. **Acórdãos 2026 — base completa**. Brasília: TCU, 2026.